# Simple Deep NN model for MNIST

In [7]:
# !pip install torch torchvision tqdm

In [8]:
import torch
import torch.nn.functional as F
from torch import nn

In [9]:
# Linear -> ReLu ... (x 4 hidden layers) -> Softmax (Output layer) (Multiclass classification)
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32) # Hidden layer 1, input_dim = flattened pixels of image inputs
        self.fc2 = nn.Linear(32, 64) # Hidden layer 2
        self.fc3 = nn.Linear(64, 128) # Hidden layer 3
        self.fc4 = nn.Linear(128, 64) # Hidden layer 4
        self.fc5 = nn.Linear(64, output_dim) # Output layer, output_dim = number of classes output

    def forward(self, x):
        x = F.relu(self.fc1(x)) # Activation after hidden layer 1
        x = F.relu(self.fc2(x)) # Activation after hidden layer 2
        x = F.relu(self.fc3(x)) # Activation after hidden layer 3
        x = F.relu(self.fc4(x)) # Activation after hidden layer 4
        x = self.fc5(x) # Output layer
        return x


# Data Loading

In [10]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [11]:
BATCH_SIZE = 64
train_dataset = datasets.MNIST(root='dataset/', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='dataset/', train=False, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
input_dim = train_dataset[0][0].numel()
no_classes = 10

# Train

In [13]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/nn_experiment")

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeuralNetwork(input_dim=input_dim, output_dim=no_classes).to(device)

/mnt/ssd2/an/projects/logisticscheduling/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [15]:
criterion = nn.CrossEntropyLoss() # Apply softmax as an activation func for the output layer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device).reshape(x.shape[0], -1)
            y = y.to(device)

            scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    model.train()

    return num_correct / num_samples

In [17]:
epochs = 5
step = 0

for epoch in range(epochs):
    running_loss = 0.0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device).reshape(data.shape[0], -1)
        targets = targets.to(device)

        # Forward propagation
        scores = model(data)
        loss = criterion(scores, targets)

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Log batch loss
        writer.add_scalar("Training Loss / Batch", loss.item(), step)

        step += 1

    # Average loss for this epoch
    avg_epoch_loss = running_loss / len(train_loader)

    # Compute accuracy after each epoch
    train_acc = check_accuracy(train_loader, model)
    test_acc = check_accuracy(test_loader, model)

    # Log epoch metrics
    writer.add_scalar("Training Loss / Epoch", avg_epoch_loss, epoch)
    writer.add_scalar("Training Accuracy", train_acc, epoch)
    writer.add_scalar("Test Accuracy", test_acc, epoch)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {avg_epoch_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Test Acc: {test_acc:.4f}"
    )

writer.close()

Epoch [1/5], Loss: 0.4909, Train Acc: 0.9310, Test Acc: 0.9300
Epoch [2/5], Loss: 0.2145, Train Acc: 0.9516, Test Acc: 0.9459
Epoch [3/5], Loss: 0.1634, Train Acc: 0.9550, Test Acc: 0.9500
Epoch [4/5], Loss: 0.1397, Train Acc: 0.9673, Test Acc: 0.9584
Epoch [5/5], Loss: 0.1191, Train Acc: 0.9696, Test Acc: 0.9610
